# Notebook 01 — Data Loading & Understanding
**Project:** Loan Default Risk Analysis and Prediction  
**Phase:** 2 of 8  
**Objective:** Load the raw dataset, understand its structure, inspect column types, examine value distributions, and document initial observations.

---

## 0. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path so src modules are importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import load_data, get_data_summary, print_data_summary
from src.config import (
    RAW_DATA_PATH, TARGET_COLUMN, ID_COLUMN,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES,
    FIGURES_DIR
)

# Plot styling
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Environment ready.')

---
## 1. Load the Dataset

In [ ]:
df = load_data(RAW_DATA_PATH)
print(f'Dataset loaded successfully: {df.shape[0]:,} rows × {df.shape[1]} columns')

---
## 2. First Look — Head, Tail, Shape

In [ ]:
print('── First 5 rows ──')
df.head()

In [ ]:
print('── Last 5 rows ──')
df.tail()

In [ ]:
print(f'Shape  : {df.shape}')
print(f'Columns: {df.columns.tolist()}')

---
## 3. Column Data Types

In [ ]:
dtype_df = pd.DataFrame({
    'Column':   df.columns,
    'Dtype':    df.dtypes.values.astype(str),
    'NonNull':  df.notnull().sum().values,
    'Null':     df.isnull().sum().values,
    'Unique':   df.nunique().values,
    'Sample':   [df[c].dropna().iloc[0] if df[c].dropna().shape[0] > 0 else 'N/A' for c in df.columns]
})
dtype_df

---
## 4. Full Dataset Summary (from `data_loader`)

In [ ]:
print_data_summary(df)

---
## 5. Target Variable — Class Distribution

In [ ]:
class_counts = df[TARGET_COLUMN].value_counts().sort_index()
class_labels = {0: 'No Default', 1: 'Default'}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
colors = ['#4a90d9', '#e05c5c']
axes[0].bar(
    [class_labels[k] for k in class_counts.index],
    class_counts.values,
    color=colors, edgecolor='white', linewidth=1.2
)
axes[0].set_title('Default Class Counts', fontweight='bold')
axes[0].set_ylabel('Number of Loans')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

# Pie chart
axes[1].pie(
    class_counts.values,
    labels=[class_labels[k] for k in class_counts.index],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
axes[1].set_title('Default Class Balance', fontweight='bold')

plt.suptitle('Target Variable Distribution — Loan Default', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'target_distribution.png', bbox_inches='tight')
plt.show()
print(f'Saved → outputs/figures/target_distribution.png')

---
## 6. Numeric Feature Distributions

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(NUMERIC_FEATURES) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURES):
    axes[i].hist(df[col].dropna(), bins=40, color='#4a90d9', edgecolor='white', linewidth=0.5)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Hide unused subplots
for j in range(len(NUMERIC_FEATURES), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'numeric_distributions.png', bbox_inches='tight')
plt.show()
print(f'Saved → outputs/figures/numeric_distributions.png')

---
## 7. Categorical Feature Distributions

In [ ]:
all_cat = CATEGORICAL_FEATURES + BINARY_FEATURES
n_cols = 3
n_rows = int(np.ceil(len(all_cat) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(all_cat):
    vc = df[col].value_counts()
    axes[i].bar(vc.index.astype(str), vc.values, color='#7c5cd8', edgecolor='white', linewidth=0.8)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    axes[i].tick_params(axis='x', rotation=30)

for j in range(len(all_cat), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'categorical_distributions.png', bbox_inches='tight')
plt.show()
print(f'Saved → outputs/figures/categorical_distributions.png')

---
## 8. Numeric Feature — Descriptive Statistics Table

In [ ]:
stats = df[NUMERIC_FEATURES].describe().T
stats['skewness'] = df[NUMERIC_FEATURES].skew().round(3)
stats['kurtosis'] = df[NUMERIC_FEATURES].kurtosis().round(3)
stats = stats.round(2)
stats

---
## 9. Unique Value Counts per Column

In [ ]:
unique_df = pd.DataFrame({
    'Column':       df.columns.tolist(),
    'UniqueValues': df.nunique().values,
    'SampleValues': [
        ', '.join(df[c].dropna().unique()[:5].astype(str)) for c in df.columns
    ]
})
unique_df

---
## 10. Observations & Notes

Fill these in after running the notebook:

| Observation | Detail |
|---|---|
| Total records | *(fill after run)* |
| Missing values | *(fill after run)* |
| Duplicate rows | *(fill after run)* |
| Default rate | *(fill after run)* |
| Class balance | *(fill after run)* |
| Notable numeric distributions | *(fill after run)* |
| Notable categorical distributions | *(fill after run)* |

---
**Next:** Notebook 02 — Data Quality Checking & Cleaning